# Du, Lai, Hu & Ye (2026) on LINK: temporal offset and context capacity

Paper doi:10.64898/2026.08.04.742876; dataset DANDI:001201 version 0.251023.2336 (the cited version;
see `02_link_development.ipynb` for the header, the full claims table and the data check).

| Claim | Status |
|---|---|
| temporal-offset | tested, readings (i) fully nested and (ii) `evaluate_decoders_nested` |
| context-capacity-contrast | tested, readings (i) and (ii) |

The release has no script for either analysis, and the paper does not name the estimator for Figure
7a or 7c; the claims file names both readings. Reading (ii) is `du2026.decoders_nested`, a port of
`evaluate_decoders_nested`, whose residual-neural gain is used.

In [ ]:
import collections, resource, time, zlib
import numpy as np
import pandas as pd
import du2026 as d

PAPER = "du-2026"
def claim_seed(claim_id):
    return zlib.crc32(f"{PAPER}/{claim_id}".encode())
T0 = time.time()
EXTRA = (1000, 2000, 3000, 4000)

assets = d.list_assets(d.LINK_ID, d.LINK_VERSION)
by_date = collections.defaultdict(list)
for a in assets:
    by_date[d.session_date(a["path"])].append(a)
paired_dates = sorted(k for k, v in by_date.items() if len(v) > 1)
dev_assets = [a for k in paired_dates for a in sorted(by_date[k], key=lambda a: a["path"])]
print(f"{len(assets)} assets in DANDI:{d.LINK_ID} {d.LINK_VERSION}; {len(paired_dates)} dates with two sessions, {len(dev_assets)} development sessions")

def table(results, name, field="gain"):
    return pd.DataFrame([{"date": r["date"], "task": r["task"], field: r[name][field]} for r in results])

In [ ]:
sessions = [d.load_link_session(a["asset_id"], a["path"], signals=("SpikingBandPower", "ThresholdCrossings")) for a in dev_assets]
check = pd.DataFrame([dict(session=s.identifier, trials=len(s.trials), bins=len(s.timestamps),
                           dt_ms=1000 * float(np.median(np.diff(s.timestamps))),
                           sbp_range=f"{s.signals['SpikingBandPower'].min():.3g}..{s.signals['SpikingBandPower'].max():.3g}",
                           tcr_range=f"{s.signals['ThresholdCrossings'].min():.3g}..{s.signals['ThresholdCrossings'].max():.3g}",
                           pos_range=f"{s.position.min():.3g}..{s.position.max():.3g}",
                           vel_range=f"{s.velocity.min():.3g}..{s.velocity.max():.3g}") for s in sessions])
print(check.to_string())
print("tasks:", collections.Counter(s.task for s in sessions), "; units:", sessions[0].units)
print("The full data check of these sessions (duplicates, clock, velocity) is in 02_link_development.ipynb.")
del sessions
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
jobs = []
# temporal offset: both histories end `offset` before the behaviour bin; one window start for all
# offsets, 280 ms after the start bin (eight SBP bins plus six offset bins; claims file)
for offset in (0, 40, 80, 120):
    build_kw = dict(signal="SpikingBandPower", context_mode="causal_history", neural_lead_ms=offset,
                    output_lead_ms=offset, sample_offset_ms=280, seed=20260726)
    jobs.append((f"offset{offset}_nested", build_kw, "nested", {"seed": 20260726}, False))
    jobs.append((f"offset{offset}_decoders", build_kw, "decoders", {"seed": 20260726}, False))
# context capacity: linear and quadratic geometry, with and without history (Methods 4.4)
for signal, tag in (("SpikingBandPower", "SBP"), ("ThresholdCrossings", "TCR")):
    for context in ("geometry_tensor", "geometry_quadratic", "causal_history", "causal_history_quadratic"):
        build_kw = dict(signal=signal, context_mode=context, seed=20260726)
        jobs.append((f"{tag}_{context}_nested", build_kw, "nested", {"seed": 20260726}, False))
        jobs.append((f"{tag}_{context}_decoders", build_kw, "decoders", {"seed": 20260726}, False))
t = time.time()
results = d.run_parallel(d.link_session_jobs, [(a["asset_id"], a["path"], jobs) for a in dev_assets])
print(f"{len(jobs)} fits per session on {len(results)} sessions in {(time.time() - t) / 60:.1f} min")
print("context feature counts are 141, 351, 151 and 361 (Methods 4.4); usable trials at the 280 ms window start:",
      sorted(set(r["offset0_nested"]["n_usable"] for r in results)))

## temporal-offset

> Shifting both the neural and the measured-output histories back by 40, 80 and 120 ms with the
> behavioral window fixed reduced the small output-history-conditional gain and changed the RD
> direction at intermediate offsets.

In [ ]:
for reading, key, field in (("(i) fully nested", "nested", "gain"), ("(ii) evaluate_decoders_nested", "decoders", "innovation_gain")):
    rows = []
    for offset in (0, 40, 80, 120):
        t_ = table(results, f"offset{offset}_{key}", field)
        rows.append(dict(offset_ms=offset, CO=t_[t_.task == "CO"][field].mean(), RD=t_[t_.task == "RD"][field].mean(),
                         RD_minus_CO=t_[t_.task == "RD"][field].mean() - t_[t_.task == "CO"][field].mean(),
                         positive=int((t_[field] > 0).sum())))
    off = pd.DataFrame(rows).set_index("offset_ms")
    print(reading)
    print(off.to_string(float_format=lambda v: f"{v:.5f}"))
    reduced = bool(off.loc[120, "CO"] < off.loc[0, "CO"] and off.loc[120, "RD"] < off.loc[0, "RD"])
    a = bool(off.loc[40, "RD"] < 0 or off.loc[80, "RD"] < 0)
    b = bool(np.sign(off.loc[40, "RD_minus_CO"]) != np.sign(off.loc[0, "RD_minus_CO"]) or np.sign(off.loc[80, "RD_minus_CO"]) != np.sign(off.loc[0, "RD_minus_CO"]))
    print(f"   gain at 120 ms below 0 ms for both tasks: {reduced}; RD change reading (a) RD mean negative at 40 or 80 ms: {a}; "
          f"reading (b) RD minus CO changes sign at 40 or 80 ms: {b}\n")

**Result: reproduced.** Under the fully nested estimator (reading i), shifting both histories from 0 to 120 ms lowers the CO mean from 0.00382 to 0.00055 and the RD mean from 0.00623 to 0.00038, and the RD-minus-CO difference turns from 0.00241 at 0 ms to -0.00024 at 40 ms and -0.00061 at 80 ms (reading b of the RD change); the RD mean stays positive (reading a fails). Under `evaluate_decoders_nested` (reading ii) the reduction holds (0.00400 to 0.00058, 0.00678 to 0.00000) and both readings of the RD change hold (RD -0.00032 at 80 ms). The outcome holds under both estimators.

## context-capacity-contrast

> Adding quadratic geometry to the context changed the descriptive RD-minus-CO gain contrast, and the
> richest quadratic-history context gave a small positive contrast for both SBP and TCR.

Per date RD minus CO, mean over nine dates, 95% paired date-bootstrap interval (10,000 resamples,
generator from the claim's seed; the seed plus 1000 to 4000 give the range of the interval bounds).

In [ ]:
for reading, key, field in (("(i) fully nested", "nested", "gain"), ("(ii) evaluate_decoders_nested", "decoders", "innovation_gain")):
    rows = []
    for tag in ("SBP", "TCR"):
        for context in ("geometry_tensor", "geometry_quadratic", "causal_history", "causal_history_quadratic"):
            p = table(results, f"{tag}_{context}_{key}", field).pivot(index="date", columns="task", values=field)
            diff = (p["RD"] - p["CO"]).to_numpy()
            bounds = []
            for k in (0,) + EXTRA:
                rng = np.random.default_rng(claim_seed("context-capacity-contrast") + k)
                boots = np.array([rng.choice(diff, len(diff), replace=True).mean() for _ in range(10000)])
                bounds.append(np.quantile(boots, [0.025, 0.975]))
            rows.append(dict(signal=tag, context=d.LADDER_NAMES[context], CO=p["CO"].mean(), RD=p["RD"].mean(), RD_minus_CO=diff.mean(),
                             ci_low=bounds[0][0], ci_high=bounds[0][1], excludes_zero=bool(bounds[0][0] > 0 or bounds[0][1] < 0),
                             excludes_zero_all_seeds=all(b[0] > 0 or b[1] < 0 for b in bounds) if (bounds[0][0] > 0 or bounds[0][1] < 0) else not any(b[0] > 0 or b[1] < 0 for b in bounds)))
    cap = pd.DataFrame(rows)
    print(reading)
    print(cap.to_string(float_format=lambda v: f"{v:.5f}"))
    qh = cap[cap.context == "quadratic plus history"].set_index("signal").RD_minus_CO
    changed = []
    for tag in ("SBP", "TCR"):
        lin = cap[(cap.signal == tag) & (cap.context == "geometry by phase")].iloc[0]
        quad = cap[(cap.signal == tag) & (cap.context == "quadratic geometry")].iloc[0]
        changed.append(bool(np.sign(lin.RD_minus_CO) != np.sign(quad.RD_minus_CO) or lin.excludes_zero != quad.excludes_zero))
    print(f"   quadratic-plus-history contrast positive: SBP {qh['SBP'] > 0}, TCR {qh['TCR'] > 0}; "
          f"quadratic contrast differs from linear (sign or interval excluding zero): SBP {changed[0]}, TCR {changed[1]}\n")
print(f"run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: reproduced.** Under the fully nested estimator the quadratic-plus-history RD-minus-CO contrast is positive for SBP (0.00374) and TCR (0.00275), and for TCR the quadratic-geometry contrast (0.00495, interval 0.00079 to 0.00915) excludes zero where the linear one (0.00227, -0.00246 to 0.00686) does not. For SBP the contrast keeps its sign and its interval contains zero in both. The release's other estimator gives the same pattern (0.00410 and 0.00374; TCR quadratic interval 0.00115 to 0.00927).

## Summary

| Claim | Outcome |
|---|---|
| temporal-offset | reproduced (under both estimators) |
| context-capacity-contrast | reproduced (under both estimators) |

The paper's figure for these analyses plots a frozen table whose script is not in the release; both
estimators the claims file names were run, and they give the same outcomes. Holding the behavioural
window fixed was read as one window start, 280 ms after the start bin, for every offset; at that start
367 to 375 trials per session are usable, against 375 in the other analyses.